# Lab 11 — Regularisation — Ridge & Lasso on Housing
**Regression Track** · Intermediate · ~55 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Demonstrate overfitting with high-degree polynomials
2. Apply Ridge (L2) and Lasso (L1) regularisation
3. Select alpha via GridSearchCV
4. Read Lasso sparsity for feature selection

## Datasets (this folder)
- `housing.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-11-regularisation-ridge-lasso/lab-11-regularisation-ridge-lasso.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`housing.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-11-regularisation-ridge-lasso"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-11-regularisation-ridge-lasso/bundle/dataset.zip"
NEED = ["housing.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Regularisation: Ridge & Lasso on California Housing

> **Scenario:** A property-valuation team fits polynomial regression to predict `median_house_value` from location and dwelling features. High-degree polynomials fit the training set beautifully and the validation set terribly. Your job: diagnose overfitting, then fix it with ridge and lasso regularisation.
>
> **You will learn:** polynomial feature expansion, the bias–variance trade-off, ridge (L2) vs lasso (L1), alpha paths, why standardisation is mandatory before regularisation.
> **Time:** ~55 minutes. **Level:** Intermediate. **Needs:** pandas + sklearn. **Env:** 🟢 Colab only.

### Mental Map

| Section | Key idea | sklearn / pandas tool |
|---|---|---|
| 1 | Load housing data; notice the capped target | `pd.read_csv`, `dropna` |
| 2 | Overfitting demo: degree 1→10 | `PolynomialFeatures` |
| 3 | Ridge: shrink all coefficients | `Ridge`, `GridSearchCV` |
| 4 | Lasso: shrink some to zero | `Lasso`, `StandardScaler` |

---

## 1. Setup and the Capped Target

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv("housing.csv")
print(df.shape)
print(df.isna().sum())

df = df.dropna(subset=["total_bedrooms"]).reset_index(drop=True)
print("after dropna:", df.shape)

print("rows at the 500k cap:", int((df["median_house_value"] >= 500000).sum()))


**What to notice:**
1. `total_bedrooms` is the only column with missing values (207 rows) — drop them.
2. The target is **capped at 500,000**: 985 rows sit exactly at the cap. The model cannot predict above it, which flattens the top of every prediction curve.
3. We use four numeric features for the polynomial demo so the feature counts stay readable.

> **Pitfall:** the cap is a data-collection artefact, not a market truth. Treat predictions above 500k with suspicion.

---

## 2. Overfitting Demo: Polynomial Degrees 1–10

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

FEATS = ["median_income", "housing_median_age", "total_rooms", "population"]
X = df[FEATS]
y = df["median_house_value"]

# Subsample for the polynomial demo — degree-3 poly on 20K rows is too slow for GridSearchCV
X = X.sample(n=3000, random_state=42)
y = y.loc[X.index]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

rows = []
for deg in range(1, 11):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=deg, include_bias=False)),
        ("lr", LinearRegression()),
    ])
    pipe.fit(X_train, y_train)
    n_feat = pipe.named_steps["poly"].n_output_features_
    tr = mean_absolute_error(y_train, pipe.predict(X_train))
    te = mean_absolute_error(y_test, pipe.predict(X_test))
    rows.append((deg, n_feat, tr, te))

res = pd.DataFrame(rows, columns=["degree", "n_features", "train_MAE", "test_MAE"])
res["overfit_gap"] = res["test_MAE"] - res["train_MAE"]
print(res.round(0))


**What to notice:**
1. Feature count explodes: degree 10 on 4 features → 1,000 columns.
2. Test MAE bottoms out at degree 5 (≈ $56,200) and then explodes — $262,600 by degree 10. Textbook overfitting.
3. The `overfit_gap` column (test − train) quantifies the damage: it first exceeds $10,000 at degree 7.

> **Pitfall:** standardising **before** `PolynomialFeatures` inside the pipeline is safe because the scaler is refit on training data only — no leakage. Fitting the scaler on the full dataset first would leak test statistics into training.

---

## 3. Ridge: Shrink Everything a Little

Ridge adds a penalty `alpha * sum(coefficients²)` — it shrinks all coefficients toward zero but never exactly to zero:

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
    ("ridge", Ridge()),
])

alphas = np.logspace(-2, 4, 5)
grid = GridSearchCV(ridge_pipe, {"ridge__alpha": alphas}, cv=5,
                    scoring="neg_mean_absolute_error", n_jobs=1)
grid.fit(X_train, y_train)

print("best alpha:", grid.best_params_["ridge__alpha"])
print("best CV MAE:", round(-grid.best_score_, 1))

cv = pd.DataFrame(grid.cv_results_)[["param_ridge__alpha", "mean_test_score"]]
cv["MAE"] = -cv["mean_test_score"]
print(cv.round(1))


**What to notice:**
1. The CV curve is U-shaped: too little alpha overfits, too much underfits. The optimum here is alpha = 100.
2. `neg_mean_absolute_error` is sklearn's convention — higher (less negative) is better.
3. Ridge keeps all 34 degree-3 features but with shrunken coefficients, and its test MAE (≈ $57,194) matches unregularised degree-3 OLS (≈ $57,145) — regularisation costs almost nothing when the model isn't overfitting.

> **Pitfall:** the best alpha depends on feature scale. If you forgot the `StandardScaler`, `median_income` (range ≈ 0–15) and `total_rooms` (range ≈ 3–39,000) would be penalised unequally.

---

## 4. Lasso: Shrink Some to Exactly Zero

Lasso penalises the sum of absolute coefficients, which drives weak features to **exactly zero** — automatic feature selection:

In [ ]:
import warnings
from sklearn.linear_model import Lasso
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

lasso_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
    ("lasso", Lasso(max_iter=10000)),
])

grid_l = GridSearchCV(lasso_pipe, {"lasso__alpha": alphas}, cv=5,
                      scoring="neg_mean_absolute_error", n_jobs=1)
grid_l.fit(X_train, y_train)

best = grid_l.best_estimator_
coefs = pd.Series(best.named_steps["lasso"].coef_,
                  index=best.named_steps["poly"].get_feature_names_out(FEATS))
nonzero = coefs[coefs != 0].sort_values(key=abs, ascending=False)
print("best alpha:", grid_l.best_params_["lasso__alpha"])
print(f"non-zero coefficients: {len(nonzero)} / {len(coefs)}")
print(nonzero.head(8).round(0))


**What to notice:**
1. At the CV-best alpha (100), lasso keeps 32 of 34 features — CV optimises **prediction**, not sparsity.
2. The survivors are interpretable: `median_income` terms dominate, as a domain expert would expect.
3. Sparsity is a separate knob: refit with `alpha=10000` and only 7 of 34 coefficients survive. Cranking alpha trades prediction for a simpler model.

> **Pitfall:** lasso with correlated features picks one arbitrarily. If two features are near-duplicates, don't read too much into which one survives.

---

## Exercises (do these!)

### Exercise 1 — Locate the overfitting knee
From the degree sweep in section 2, find the degree with the **lowest test MAE** and the degree where the overfit gap first exceeds $10,000.
*Expected: best degree = 5; gap first exceeds $10k at degree 7.*

**Follow-up:** How many features does degree 10 create? Check: 1000.

### Exercise 2 — Best ridge alpha
Run the ridge grid search and report the best alpha and its CV MAE.
*Expected: alpha = 100, CV MAE ≈ $56,593.*

**Follow-up:** Does ridge beat unregularised degree-10 OLS on test MAE? Check: yes — $57,194 vs $262,600.

### Exercise 3 — Lasso sparsity
Report how many of the 34 degree-3 coefficients are non-zero at lasso's best alpha, and name the top feature.
*Expected: 32 / 34 non-zero; top feature `median_income`.*

**Follow-up:** Refit lasso with `alpha=10000` — how many survive? Check: 7.

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
# --- Solution 1 ---
best_deg = res.loc[res["test_MAE"].idxmin(), "degree"]
first_big_gap = res.loc[res["overfit_gap"] > 10000, "degree"].min()
print(f"best degree: {best_deg}, first gap>10k at degree: {first_big_gap}")
print("degree-10 feature count:", int(res.loc[res.degree == 10, "n_features"].iloc[0]))

# --- Solution 2 ---
print("best ridge alpha:", grid.best_params_["ridge__alpha"])
print("best ridge CV MAE:", round(-grid.best_score_, 1))
ols3 = Pipeline([("scaler", StandardScaler()),
                 ("poly", PolynomialFeatures(degree=3, include_bias=False)),
                 ("lr", LinearRegression())]).fit(X_train, y_train)
ols3_test = mean_absolute_error(y_test, ols3.predict(X_test))
ridge_test = mean_absolute_error(y_test, grid.best_estimator_.predict(X_test))
print(f"OLS deg-3 test MAE: {ols3_test:.1f}")
print(f"ridge test MAE: {ridge_test:.1f}")
print(f"OLS deg-10 test MAE: {res.loc[res.degree == 10, 'test_MAE'].iloc[0]:.1f}")

# --- Solution 3 ---
print("non-zero lasso coefs:", len(nonzero), "/", len(coefs))
print("top feature:", nonzero.abs().idxmax())
lasso_hard = Pipeline([("scaler", StandardScaler()),
                       ("poly", PolynomialFeatures(degree=3, include_bias=False)),
                       ("lasso", Lasso(alpha=10000, max_iter=10000))]).fit(X_train, y_train)
nz_hard = int((lasso_hard.named_steps["lasso"].coef_ != 0).sum())
print(f"non-zero at alpha=10000: {nz_hard} / {len(coefs)}")

# --- Follow-up 1 ---
assert int(res.loc[res.degree == 10, "n_features"].iloc[0]) == 1000

# --- Follow-up 2 ---
assert ridge_test < res.loc[res.degree == 10, "test_MAE"].iloc[0]
print(f"ridge beats deg-10 OLS by {res.loc[res.degree == 10, 'test_MAE'].iloc[0] - ridge_test:.1f}")

# --- Follow-up 3 ---
assert nz_hard < len(coefs), "lasso should zero out some coefficients"
print(f"at alpha=10000, {nz_hard} of {len(coefs)} coefficients survive")


### What to learn next
- ElasticNet blends L1 and L2 — try `ElasticNet(l1_ratio=0.5)`.
- Plot the full coefficient path (coefficients vs alpha) to see shrinkage visually.
- Move to `RidgeCV` / `LassoCV`, which choose alpha internally with efficient path algorithms.

*Files in this folder: housing.csv.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
